# 2.2 - Modeling - Fine Tunning

This step is a continuation of the previous one, **2.1 - Modeling.ipynb**, which was used to define the best model for the analysis in question based on the criteria of time and importance to the business, which in this case would correspond to `recall` and `test_average_precision`.

From now on, **Fine Tuning** will be performed to obtain the best model, then export it using the `joblib` library and then apply it to the tests in section **2.3 - Modeling_Test**

## Importing Libs

In [1]:
# Importing Libs
    
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="darkgrid", rc={'figure.figsize':(10,6)})

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report,accuracy_score, recall_score

from xgboost import XGBClassifier

import warnings

## Importing Dataset and Feature Engeneering

In this step:
- The dataset was imported
- The values created in section **1 - EDA.ipynb**, `balanceOrigDiff` and `balanceDestDiff` were used
- Values that would not be relevant in the forecast were removed

In [3]:
FRAUD_PATH = "datasets/AIML Dataset.csv"

df = pd.read_csv(FRAUD_PATH)

categorical_features = ['type', 'nameOrig', 'nameDest', 'isFraud', 'isFlaggedFraud']
numerical_features = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

df['balanceOrigDiff'] = df['newbalanceOrig'] - df['oldbalanceOrg']
df['balanceDestDiff'] = df['newbalanceDest'] - df['oldbalanceDest']

features_model = numerical_features + ['balanceOrigDiff', 'balanceDestDiff', 'isFraud']
df_model = df[features_model]

df_model.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,balanceOrigDiff,balanceDestDiff,isFraud
0,1,9839.64,170136.0,160296.36,0.0,0.0,-9839.64,0.0,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,-1864.28,0.0,0
2,1,181.00,181.0,0.00,0.0,0.0,-181.00,0.0,1
3,1,181.00,181.0,0.00,21182.0,0.0,-181.00,-21182.0,1
4,1,11668.14,41554.0,29885.86,0.0,0.0,-11668.14,0.0,0


## Train Test Split

The data was divided as follows:
- The class of interest (`isFraud`) was separated as **y** and the others as **X**
- The division was made in a 70-30 ratio due to the high volume of data
- The `stratify` hyperparameter was used in order to maintain the proportion of positive and negative classes for both the test set and the training set
- Since there are no categorical values, it was only necessary to make the `StandardScaler`

In [4]:
y = df_model["isFraud"]
X = df_model.drop("isFraud", axis = 1)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, stratify=y, random_state = 42)

In [6]:
scaler =  StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

`StratifiedKFold` is used to divide the data into K parts (folds) to perform cross-validation, maintaining the proportion of classes in each fold.
In other words, in each Cross-Validation it is guaranteed that each fold has representation of the classes.

In [7]:
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Fine Tunning

The parameters will be subjected to Fine Tuning through `GridSearchCV`, where in each iteration for values ​​that will be tested the model will be subjected to a k-fold to avoid possible overfitting problems. In addition, the following parameters will be highlighted:

- The same metrics that were used to evaluate the previous models will be implemented.
- The model chosen was the **XGB Classifier**
- 5 hyperparameters when subjected to Tunning, which will have two variations each (in previous tests other hyperparameters were tested, but due to the test time only the ones that had the best performance will be tested)
- The determining metric expressed by `refit` was the `"average_precision"`, or AUPRC.

In [8]:
MODEL_EVALUATION_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "f1",
    "precision",
    "recall",
    "roc_auc",
    "average_precision",
    "neg_brier_score",
    "f1_weighted",
]

In [11]:
# Model Instance
xgb = XGBClassifier(use_label_encoder=False, 
                    eval_metric='logloss', 
                    random_state=42)

# Hyperparameters Dict
param_grid = {
    'n_estimators': [350, 400],
    'max_depth': [9, 12],
    'learning_rate': [0.1, 0.3],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.9, 1]
}

In [12]:
# GridSearch Instances
grid_search = GridSearchCV(
    estimator = xgb,
    param_grid = param_grid,
    scoring = MODEL_EVALUATION_METRICS,
    refit="average_precision",
    cv = stratified_kfold,
    n_jobs = 1,
    verbose = 2
)

In [19]:
# Fitting the model with training data
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    
    grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.8; total time=  54.8s
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.8; total time=  59.0s
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.8; total time=  56.7s
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.8; total time=  58.2s
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.8; total time=  59.0s
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.9; total time= 1.1min
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.9; total time= 1.1min
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.9; total time=  57.2s
[CV] END colsample

In [141]:
# Results
print("best parameters:", grid_search.best_params_)
print("best average_precision score:", grid_search.best_score_)

best parameters: {'colsample_bytree': 1, 'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 400, 'subsample': 0.9}
best average_precision score: 0.9377652170875471


## Evaluating Model

After obtaining the best parameters for the model, the scores obtained through `GridSearchCV` will then be evaluated:

In [127]:
def best_estimator_gs_mean_split_values(GRID_SEARCH: GridSearchCV) -> pd.DataFrame:
    """
    Get mean and splits scores for the best estimator from a GridSearchCV object.

    Parameters
    ----------
    GRID_SEARCH : GridSearchCV
        Fitted GridSearchCV object.

    Returns
    -------
    pd.DataFrame
        DataFrame with mean and splits scores for the best estimator
    """
    
    best_estimator_index = GRID_SEARCH.best_index_
    cv_results = GRID_SEARCH.cv_results_

    mean_scores = {}
    split_scores = {f"split{i}": {} for i in range(5)}

    # Iterar por cada métrica
    for metric in GRID_SEARCH.scoring:
        mean_scores[metric] = cv_results[f"mean_test_{metric}"][best_estimator_index]

        for i in range(5):
            split_scores[f"split{i}"][metric] = cv_results[f"split{i}_test_{metric}"][best_estimator_index]


    df_mean_scores = pd.DataFrame(mean_scores, index=["mean_score"]).T
    df_split_scores = pd.DataFrame(split_scores).T 
    df_split_scores.index.name = "split"

    return df_mean_scores, df_split_scores

In [147]:
df_mean, df_split = get_mean_and_std_from_grid_search_best_estimator(grid_search)

In [149]:
df_mean

,mean_score
accuracy,0.999683
balanced_accuracy,0.907422
f1,0.869020
precision,0.930856
recall,0.814921
roc_auc,0.999287
average_precision,0.937765
neg_brier_score,-0.000251
f1_weighted,0.999672


In [151]:
df_split

,accuracy,balanced_accuracy,f1,precision,recall,roc_auc,average_precision,neg_brier_score,f1_weighted
split,,,,,,,,,
split0,0.999689,0.910395,0.872055,0.930049,0.820870,0.999312,0.942226,-0.000241,0.999679
split1,0.999679,0.907784,0.867715,0.926877,0.815652,0.999355,0.937469,-0.000254,0.999669
split2,0.999695,0.911700,0.874423,0.932087,0.823478,0.998619,0.938549,-0.000241,0.999685
split3,0.999696,0.908227,0.873895,0.939940,0.816522,0.999622,0.938762,-0.000248,0.999685
split4,0.999656,0.899001,0.857009,0.925328,0.798085,0.999528,0.931821,-0.000271,0.999644


## Saving Model

The best model will then be saved to be used in the next steps:

In [29]:
import joblib
filename = 'best_xgb_model.joblib'

joblib.dump(grid_search.best_estimator_, filename)

['best_xgb_model.joblib']